<a href="https://colab.research.google.com/github/siwitz15/lab-ai-69/blob/main/Stronger_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 💪 Stronger Assistant — โค้ชออกกำลังกายส่วนตัว

แชทบอทที่ **ถามข้อมูลของคุณก่อน** (น้ำหนัก อายุ ระดับ อุปกรณ์ จำนวนวันที่ว่าง อาการบาดเจ็บ) แล้วค่อยแนะนำ
ท่าออกกำลัง · อุปกรณ์ · น้ำหนักที่ควรยก · กี่เซ็ตกี่ครั้ง · เวลาที่ใช้ · วันไหนเล่นอะไร

สร้างตามโครงเดียวกับ "ห้องทดลองของบิ๊บ" ของอาจารย์ โดยจับคู่แนวคิดดังนี้

| แนวคิดในห้องทดลองบิ๊บ | ใน Stronger Assistant |
|---|---|
| ซองจดหมาย (`build_envelope`) | โค้ชได้รับกฎ + โปรไฟล์ + แผน + การ์ดท่า ทุกครั้งที่ถาม |
| สมุดบันทึกข้างตัว (structured memory) | โปรไฟล์ผู้ใช้ที่ดึงจากบทสนทนาเป็น JSON |
| เครื่องมือ (tool use) | ฟังก์ชัน Python คำนวณน้ำหนัก เซ็ต เวลา และจัดตารางรายสัปดาห์ |
| RAG / การ์ดจากคู่มือ | ฐานความรู้ท่าออกกำลัง กรองด้วย metadata (กล้ามเนื้อ อุปกรณ์ ระดับ อาการบาดเจ็บ) |
| ข้อสอบชุดทอง | เซลล์ทดสอบว่าบอท "ถามก่อนแนะนำ" จริง |

**หลักออกแบบสำคัญ:** ตัวเลขทุกตัว (กก. เซ็ต ครั้ง นาที) คำนวณด้วยโค้ด ไม่ให้ LLM เดา LLM มีหน้าที่คุยภาษาคนและอธิบายเท่านั้น

**วิธีใช้:** กด **+ Code** วางทีละเซลล์ รัน `Shift+Enter` ตามลำดับ ห้ามข้ามขั้น (หรือเปิดไฟล์นี้ใน Colab แล้ว Runtime → Run all)

## ขั้น 0 ติดตั้งและต่อสายถึงสมอง

ขอกุญแจฟรีที่ [Google AI Studio](https://aistudio.google.com/apikey) แล้วใส่ใน Colab → ไอคอนกุญแจ (**Secrets**) ตั้งชื่อ `GEMINI_API_KEY` แล้วเปิด **Notebook access**

In [1]:
!pip -q install google-genai gradio

In [2]:
import os, re, json, time, math
from google import genai
from google.genai import errors
from google.colab import userdata

MODEL = "gemini-3.5-flash-lite"   # ถ้า error ว่าไม่รู้จักรุ่น ให้แก้เป็นรุ่นที่บัญชีคุณใช้ได้
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

def to_gemini(messages, temperature):
    system = "\n\n".join(m["content"] for m in messages if m["role"] == "system") or None
    contents = [{"role": "user" if m["role"] == "user" else "model", "parts": [{"text": m["content"]}]}
                for m in messages if m["role"] != "system"]
    return contents, {"system_instruction": system, "temperature": temperature}

def call_llm(messages, temperature=0.3):
    """ส่งซองให้โค้ช แล้วรับคำตอบเป็นข้อความ (ถ้าโควตาฟรีเต็ม จะรอแล้วลองใหม่)"""
    contents, config = to_gemini(messages, temperature)
    for wait in [10, 20, 40, 0]:
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config).text or ""
        except errors.APIError as e:
            if e.code != 429 or wait == 0:
                raise
            print(f"โควตาเต็ม รอ {wait} วินาที"); time.sleep(wait)

def build_envelope(question, system_rules, memory="", cards="", history=None):
    """กฎประจำตัว → สมุดบันทึก(โปรไฟล์) → การ์ดความรู้ → ประวัติแชท → คำถาม"""
    system = system_rules
    if memory:
        system += "\n\n[โปรไฟล์ผู้ใช้]\n" + memory
    if cards:
        system += "\n\n[ข้อมูลอ้างอิง]\n" + cards
    return [{"role": "system", "content": system}] + (history or []) + [{"role": "user", "content": question}]

print(call_llm([{"role": "user", "content": "สวัสดี แนะนำตัวหนึ่งประโยคในฐานะโค้ชฟิตเนส"}]))

สวัสดีครับ ผมโค้ช [ชื่อของคุณ] พร้อมผลักดันคุณให้ก้าวข้ามขีดจำกัดและปั้นหุ่นในฝันไปด้วยกันครับ!


## ขั้น 1 ฐานความรู้ท่าออกกำลัง (การ์ดแบบ RAG)

ท่าแต่ละท่าคือหนึ่ง "การ์ด" มี metadata ให้กรองได้ ได้แก่ กล้ามเนื้อที่เน้น · อุปกรณ์ · ระดับต่ำสุดที่ทำได้ · ท่านี้ควรเลี่ยงถ้าบาดเจ็บที่ไหน
และ `ratio` = น้ำหนักเริ่มต้นของ**มือใหม่**เป็นสัดส่วนของน้ำหนักตัว (ดัมเบลนับเป็นต่อข้าง) ใช้คำนวณน้ำหนักที่เหมาะกับแต่ละคน

> ตัวเลข ratio เป็นค่าประมาณทั่วไปสำหรับจุดเริ่มต้น ไม่ใช่ค่ามาตรฐานทางการแพทย์ ปรับตามที่เทรนเนอร์หรือประสบการณ์จริงได้เลย

In [3]:
def E(id, th, en, muscle, equip, level, ratio, iso, avoid, cue, unit="rep"):
    return dict(id=id, th=th, en=en, muscle=muscle, equip=equip, level=level,
                ratio=ratio, iso=iso, avoid=avoid, cue=cue, unit=unit)

EXERCISES = [
    # ---- อก ----
    E("machine_chest", "เครื่องเชสต์เพรส", "Machine Chest Press", "chest", "machine", 1, 0.35, False, ["shoulder"],
      "ปรับเบาะให้มือเสมอกลางอก ดันออกโดยไม่ยกไหล่ ปล่อยกลับช้า ๆ"),
    E("db_bench", "ดัมเบลเบนช์เพรส", "Dumbbell Bench Press", "chest", "dumbbell", 1, 0.15, False, ["shoulder"],
      "นอนบนม้านั่ง (ไม่มีม้านั่งนอนบนพื้นได้) ลดดัมเบลช้า ๆ ให้ศอกทำมุมราว 45° จากลำตัว แล้วดันขึ้น"),
    E("bb_bench", "บาร์เบลเบนช์เพรส", "Barbell Bench Press", "chest", "barbell", 2, 0.45, False, ["shoulder", "wrist"],
      "สะบักหุบเข้าหากัน เท้าแนบพื้น ลดบาร์แตะกลางอกเบา ๆ แล้วดันขึ้น ควรมีคนสปอตต์"),
    E("pushup", "วิดพื้น", "Push-up", "chest", "bodyweight", 1, None, False, ["wrist", "shoulder"],
      "ลำตัวตรงเป็นแนวเดียว ลดอกลงเกือบถึงพื้น ถ้ายังไม่ไหวให้วิดเข่าหรือวางมือบนที่สูง"),
    E("cable_fly", "เคเบิลฟลาย", "Cable Fly", "chest", "cable", 2, 0.06, True, ["shoulder"],
      "ศอกงอเล็กน้อยคงที่ ดึงมือสองข้างมาบรรจบกลางอก แล้วกางออกช้า ๆ"),
    # ---- หลัง ----
    E("lat_pull", "แลทพูลดาวน์", "Lat Pulldown", "back", "machine", 1, 0.5, False, [],
      "นั่งให้ต้นขาล็อกใต้แป้น ดึงบาร์ลงมาที่ไหปลาร้าโดยใช้ศอกนำ ไม่เอนหลังมาก"),
    E("seated_row", "ซีทเต็ดเคเบิลโรว์", "Seated Cable Row", "back", "cable", 1, 0.45, False, [],
      "หลังตรง ดึงมือเข้าหาท้อง บีบสะบักเข้าหากัน ปล่อยกลับช้า ๆ"),
    E("db_row", "ดัมเบลโรว์แขนเดียว", "One-Arm Dumbbell Row", "back", "dumbbell", 1, 0.15, False, [],
      "มือหนึ่งยันม้านั่ง หลังขนานพื้น ดึงดัมเบลเข้าหาสะโพก บีบสะบัก"),
    E("bb_row", "บาร์เบลโรว์", "Barbell Row", "back", "barbell", 2, 0.4, False, ["back"],
      "โน้มตัวหลังตรง ดึงบาร์เข้าหาท้องน้อย อย่าเหวี่ยงตัว"),
    E("inv_row", "ดึงตัวใต้โต๊ะ/บาร์ต่ำ", "Inverted Row", "back", "bodyweight", 1, None, False, [],
      "นอนใต้โต๊ะที่แข็งแรงหรือบาร์ต่ำ จับขอบ ดึงอกเข้าหา ลำตัวตรงเป็นแนวเดียว"),
    E("superman", "ซูเปอร์แมน", "Superman", "back", "bodyweight", 1, None, True, ["back"],
      "นอนคว่ำ ยกแขนและขาขึ้นพร้อมกันค้างสั้น ๆ แล้วลง ไม่เงยคอ"),
    # ---- ไหล่ ----
    E("machine_shoulder", "เครื่องช็อลเดอร์เพรส", "Machine Shoulder Press", "shoulders", "machine", 1, 0.3, False, ["shoulder"],
      "หลังแนบเบาะ ดันขึ้นเหนือศีรษะ ไม่แอ่นหลัง"),
    E("db_press", "ดัมเบลช็อลเดอร์เพรส", "Dumbbell Shoulder Press", "shoulders", "dumbbell", 1, 0.12, False, ["shoulder"],
      "นั่งหลังตรง ดันดัมเบลขึ้นเหนือศีรษะ ท้องเกร็ง ไม่แอ่นหลัง"),
    E("lat_raise", "ดัมเบลแลทเทอรัลเรส", "Dumbbell Lateral Raise", "shoulders", "dumbbell", 1, 0.04, True, ["shoulder"],
      "ยกแขนออกข้างถึงระดับไหล่ ศอกงอเล็กน้อย ใช้น้ำหนักเบา ไม่แกว่งตัว"),
    E("face_pull", "เฟซพูล", "Face Pull", "shoulders", "cable", 2, 0.12, True, [],
      "ดึงเชือกเข้าหาใบหน้า ศอกสูงระดับไหล่ บีบสะบัก ช่วยสมดุลไหล่"),
    E("pike", "ไพค์พุชอัพ", "Pike Push-up", "shoulders", "bodyweight", 2, None, False, ["shoulder", "wrist"],
      "ตั้งสะโพกสูงเป็นรูปตัว V ลดศีรษะลงไปหาพื้นแล้วดันขึ้น"),
    # ---- ต้นแขนหน้า (ไบเซ็ป) ----
    E("db_curl", "ดัมเบลเคิร์ล", "Dumbbell Curl", "biceps", "dumbbell", 1, 0.10, True, ["elbow"],
      "ยืนตัวตรง ศอกแนบลำตัว ยกดัมเบลขึ้นโดยไม่แกว่งตัว ลดลงช้า ๆ ประมาณ 2–3 วินาที"),
    E("hammer", "แฮมเมอร์เคิร์ล", "Hammer Curl", "biceps", "dumbbell", 1, 0.10, True, ["elbow"],
      "ถือดัมเบลแนวตั้ง (ฝ่ามือเข้าหากัน) ยกขึ้นโดยศอกอยู่กับที่ เน้นกล้ามเนื้อต้นแขนด้านนอกและปลายแขน"),
    E("cable_curl", "เคเบิลเคิร์ล", "Cable Curl", "biceps", "cable", 1, 0.20, True, ["elbow"],
      "ศอกแนบลำตัว งอแขนยกขึ้นเต็มช่วง ลดลงช้า ๆ แรงต้านคงที่ตลอดท่า"),
    E("bb_curl", "บาร์เบลเคิร์ล", "Barbell Curl", "biceps", "barbell", 2, 0.25, True, ["wrist", "elbow"],
      "ยืนตรง ศอกแนบลำตัว ยกบาร์ขึ้นโดยไม่เหวี่ยงหลัง"),
    E("inc_curl", "อินไคลน์ดัมเบลเคิร์ล", "Incline Dumbbell Curl", "biceps", "dumbbell", 2, 0.07, True, ["elbow"],
      "นั่งเอนบนม้านั่งเอียง แขนห้อยตรง ยกขึ้นช้า ๆ ได้ช่วงยืดของไบเซ็ปมาก"),
    E("towel_curl", "เคิร์ลด้วยผ้าขนหนู", "Towel Curl", "biceps", "bodyweight", 1, None, True, ["elbow"],
      "ยืนเหยียบกลางผ้าขนหนู จับปลายสองข้าง ดึงขึ้นเหมือนเคิร์ลพร้อมกดเท้าต้านไว้ให้ผ้าตึง"),
    # ---- ต้นแขนหลัง (ไตรเซ็ป) ----
    E("pushdown", "เคเบิลพุชดาวน์", "Triceps Pushdown", "triceps", "cable", 1, 0.25, True, ["elbow"],
      "ศอกแนบลำตัว กดลงจนแขนตรง ไม่ยกไหล่"),
    E("oh_tri", "ดัมเบลโอเวอร์เฮดไตรเซ็ปเอ็กซ์เทนชัน", "Overhead Dumbbell Triceps Extension", "triceps", "dumbbell", 1, 0.15, True, ["elbow", "shoulder"],
      "ถือดัมเบลอันเดียวสองมือเหนือศีรษะ งอศอกลดไปหลังศีรษะ แล้วเหยียดขึ้น ศอกชี้ขึ้นไม่กางออก"),
    E("close_pushup", "วิดพื้นมือชิด", "Close-grip Push-up", "triceps", "bodyweight", 1, None, False, ["wrist", "elbow"],
      "วางมือแคบกว่าไหล่ ศอกแนบลำตัวตอนลง"),
    E("bench_dip", "ดิปเก้าอี้", "Bench Dip", "triceps", "bodyweight", 2, None, False, ["shoulder", "wrist"],
      "มือยันเก้าอี้ด้านหลัง งอศอกลดตัวช่วงสั้น ๆ ไม่ให้ไหล่ยกขึ้น"),
    # ---- ต้นขาหน้า ----
    E("leg_press", "เลกเพรส", "Leg Press", "quads", "machine", 1, 1.0, False, ["knee"],
      "วางเท้าความกว้างเท่าไหล่ ลดแป้นจนเข่างอราว 90° ดันขึ้นโดยไม่ล็อกเข่า"),
    E("bb_squat", "บาร์เบลสควอท", "Barbell Back Squat", "quads", "barbell", 2, 0.6, False, ["knee", "back"],
      "บาร์พาดบนหลังส่วนบน หลังตรง นั่งลงจนต้นขาขนานพื้น ควรมีสปอตต์หรือใช้แร็ก"),
    E("goblet", "ก็อบเล็ตสควอท", "Goblet Squat", "quads", "dumbbell", 1, 0.20, False, ["knee"],
      "ถือดัมเบลอันเดียวที่หน้าอก นั่งลงให้สะโพกไปด้านหลัง เข่าชี้ตามปลายเท้า ส้นเท้าแนบพื้น"),
    E("db_lunge", "ดัมเบลลันจ์", "Dumbbell Lunge", "quads", "dumbbell", 2, 0.12, False, ["knee"],
      "ก้าวยาว ลดเข่าหลังเกือบแตะพื้น ลำตัวตรง ดันกลับด้วยส้นเท้าหน้า"),
    E("bw_squat", "สควอทน้ำหนักตัว", "Bodyweight Squat", "quads", "bodyweight", 1, None, False, ["knee"],
      "เท้ากว้างเท่าไหล่ นั่งลงให้สะโพกไปด้านหลัง หลังตรง ยืดแขนไปด้านหน้าช่วยทรงตัว"),
    E("bw_lunge", "ลันจ์น้ำหนักตัว", "Bodyweight Lunge", "quads", "bodyweight", 1, None, False, ["knee"],
      "ก้าวยาวไปด้านหน้า ลดเข่าหลังเกือบแตะพื้น แล้วดันกลับ สลับข้าง"),
    E("leg_ext", "เลกเอ็กซ์เทนชัน", "Leg Extension", "quads", "machine", 1, 0.25, True, ["knee"],
      "เหยียดเข่าจนตรงแล้วค้างครู่ ลดลงช้า ๆ ไม่เหวี่ยง"),
    # ---- ต้นขาหลัง/สะโพก ----
    E("leg_curl", "ไลอิ้งเลกเคิร์ล", "Lying Leg Curl", "hamstrings", "machine", 1, 0.25, True, ["knee"],
      "งอเข่าดึงส้นเท้าเข้าหาก้น สะโพกแนบเบาะ ลดลงช้า ๆ"),
    E("rdl_db", "ดัมเบลรูมาเนียนเดดลิฟต์", "Dumbbell Romanian Deadlift", "hamstrings", "dumbbell", 1, 0.15, False, ["back"],
      "ถือดัมเบลสองข้าง ดันสะโพกไปด้านหลัง หลังตรงตลอด ลดดัมเบลตามหน้าขาจนรู้สึกตึงต้นขาหลัง"),
    E("rdl_bb", "บาร์เบลรูมาเนียนเดดลิฟต์", "Barbell Romanian Deadlift", "hamstrings", "barbell", 2, 0.4, False, ["back"],
      "บาร์ชิดขา สะโพกไปด้านหลัง หลังตรง ลดจนรู้สึกตึงต้นขาหลัง"),
    E("bridge", "กลูทบริดจ์", "Glute Bridge", "hamstrings", "bodyweight", 1, None, False, [],
      "นอนหงายชันเข่า ดันสะโพกขึ้นจนลำตัวเป็นเส้นตรง บีบก้นค้าง 1 วินาที"),
    # ---- น่อง ----
    E("calf_machine", "เครื่องยกน่อง", "Machine Calf Raise", "calves", "machine", 1, 0.5, True, [],
      "ยกส้นเท้าสุดแล้วค้าง 1 วินาที ลดลงช้า ๆ จนยืดเต็มช่วง"),
    E("calf_bw", "ยกส้นเท้า", "Calf Raise", "calves", "bodyweight", 1, None, True, [],
      "ยืนปลายเท้าบนขอบขั้นบันได ยกส้นสุดแล้วลดลงช้า ๆ"),
    # ---- แกนกลางลำตัว ----
    E("plank", "แพลงก์", "Plank", "core", "bodyweight", 1, None, True, [],
      "ศอกใต้ไหล่ ลำตัวตรง ท้องเกร็ง ไม่แอ่นหลังหรือยกก้นสูง", unit="sec"),
    E("dead_bug", "เดดบัก", "Dead Bug", "core", "bodyweight", 1, None, True, [],
      "นอนหงาย ยกแขนขาขึ้น เหยียดแขนกับขาตรงข้ามออกสลับกัน หลังส่วนล่างแนบพื้น"),
    E("cable_crunch", "เคเบิลครันช์", "Cable Crunch", "core", "cable", 2, 0.20, True, ["back"],
      "คุกเข่า ม้วนลำตัวลงด้วยกล้ามเนื้อหน้าท้อง ไม่ใช้สะโพกดึง"),
    E("hang_knee", "แขวนยกเข่า", "Hanging Knee Raise", "core", "bodyweight", 2, None, True, ["shoulder"],
      "แขวนบาร์ ยกเข่าขึ้นหาอกโดยไม่แกว่งตัว"),
]

MUSCLE_TH = {"chest": "อก", "back": "หลัง", "shoulders": "ไหล่", "biceps": "ต้นแขนหน้า (ไบเซ็ป)",
             "triceps": "ต้นแขนหลัง (ไตรเซ็ป)", "quads": "ต้นขาหน้า", "hamstrings": "ต้นขาหลัง/สะโพก",
             "calves": "น่อง", "core": "แกนกลางลำตัว"}
EQUIP_TH = {"dumbbell": "ดัมเบล", "barbell": "บาร์เบล", "machine": "เครื่อง", "cable": "สายเคเบิล", "bodyweight": "น้ำหนักตัว"}
print(f"ฐานความรู้มี {len(EXERCISES)} ท่า ครอบคลุม {len(MUSCLE_TH)} กลุ่มกล้ามเนื้อ")

ฐานความรู้มี 43 ท่า ครอบคลุม 9 กลุ่มกล้ามเนื้อ


## ขั้น 2 เครื่องมือคำนวณแผน (ตัวเลขทุกตัวมาจากโค้ด)

เครื่องมือชุดนี้ทำ 4 อย่าง
1. **คำนวณน้ำหนักเริ่มต้น** จาก น้ำหนักตัว × ระดับ × เพศ × อายุ × เป้าหมาย
2. **กำหนดเซ็ต/ครั้ง/เวลาพัก** ตามเป้าหมาย (เพิ่มกล้าม 8–12 ครั้ง · ความแข็งแรง 4–6 ครั้ง · ลดไขมัน 12–15 ครั้ง) และเพิ่ม 1 เซ็ตให้กล้ามเนื้อที่ผู้ใช้อยากเน้น
3. **คำนวณเวลา** ต่อท่าและต่อวัน
4. **จัดตารางรายสัปดาห์** ตามจำนวนวันที่ว่าง พร้อมกรองท่าที่ไม่เหมาะกับระดับ อุปกรณ์ และอาการบาดเจ็บ

In [4]:
LEVEL_NUM = {"beginner": 1, "intermediate": 2, "advanced": 3}
LEVEL_MULT = {1: 1.0, 2: 1.4, 3: 1.8}
SEX_MULT = {"male": 1.0, "female": 0.7}
GOAL_MULT = {"muscle": 1.0, "strength": 1.1, "fat_loss": 0.85}
EQUIP_OK = {"gym": {"dumbbell", "barbell", "machine", "cable", "bodyweight"},
            "dumbbell": {"dumbbell", "bodyweight"},
            "bodyweight": {"bodyweight"}}
SINGLE_DB = {"goblet", "oh_tri"}          # ท่าที่ถือดัมเบลอันเดียว ที่เหลือคือต่อข้าง

def suggest_load(ex, p):
    """น้ำหนักเริ่มต้น (กก.) ของท่านี้สำหรับผู้ใช้คนนี้ คืน None ถ้าเป็นท่าน้ำหนักตัว"""
    if ex["ratio"] is None:
        return None
    m = ex["ratio"] * p["weight_kg"] * LEVEL_MULT[LEVEL_NUM[p["level"]]] * SEX_MULT[p["sex"]]
    if not ex["iso"]:
        m *= GOAL_MULT[p["goal"]]
    if p["age"] >= 50: m *= 0.85
    if p["age"] < 18: m *= 0.8
    step = 1 if ex["equip"] == "dumbbell" else 2.5
    low = 20 if ex["equip"] == "barbell" else step      # บาร์เบลเปล่าหนัก 20 กก.
    return max(low, round(m / step) * step)

def prescription(ex, p, focus):
    """คืน (เซ็ต, (ครั้งต่ำ, ครั้งสูง), วินาทีพัก)"""
    g = p["goal"]
    if g == "strength" and not ex["iso"]:
        sets, reps, rest = 4, (4, 6), 150
    elif g == "fat_loss":
        sets, reps, rest = 3, (12, 15), 45
    else:
        sets, reps, rest = 3, (8, 12), 75
    if p["level"] == "beginner":
        sets = min(sets, 3)
    if ex["muscle"] in focus:
        sets = min(sets + 1, 5)              # กล้ามเนื้อที่อยากเน้น +1 เซ็ต
    if ex["unit"] == "sec":
        reps, rest = (20, 45), 45
    return sets, reps, rest

def minutes_for(ex, sets, reps, rest):
    work = (reps[0] + reps[1]) / 2 * (1 if ex["unit"] == "sec" else 4)   # ครั้งละราว 4 วินาที
    return sets * (work + rest) / 60

DAY_TYPES = {
    "FULL_A": ("ทั้งตัว A", ["quads", "chest", "back", "shoulders", "core"]),
    "FULL_B": ("ทั้งตัว B", ["hamstrings", "chest", "back", "biceps", "triceps"]),
    "PUSH":   ("ดัน (อก ไหล่ หลังแขน)", ["chest", "chest", "shoulders", "triceps"]),
    "PULL":   ("ดึง (หลัง หน้าแขน)", ["back", "back", "biceps", "core"]),
    "LEGS":   ("ขา", ["quads", "hamstrings", "quads", "calves", "core"]),
    "UPPER":  ("ลำตัวส่วนบน", ["chest", "back", "shoulders", "biceps", "triceps"]),
    "LOWER":  ("ลำตัวส่วนล่าง", ["quads", "hamstrings", "quads", "calves", "core"]),
}
SPLITS = {2: ["FULL_A", "FULL_B"], 3: ["PUSH", "PULL", "LEGS"],
          4: ["UPPER", "LOWER", "UPPER", "LOWER"], 5: ["PUSH", "PULL", "LEGS", "UPPER", "LOWER"]}
WEEK = ["จันทร์", "อังคาร", "พุธ", "พฤหัสบดี", "ศุกร์", "เสาร์", "อาทิตย์"]
TRAIN_DAYS = {2: [0, 3], 3: [0, 2, 4], 4: [0, 1, 3, 4], 5: [0, 1, 2, 3, 4]}

def pick_exercise(muscle, p, used_today, used_count):
    """หยิบท่าจากฐานความรู้ โดยกรองด้วย metadata แล้วสลับท่าไม่ให้ซ้ำกันทุกวัน"""
    bad = set(p["injuries"])
    ok = [e for e in EXERCISES
          if e["muscle"] == muscle and e["equip"] in EQUIP_OK[p["equipment"]]
          and e["level"] <= LEVEL_NUM[p["level"]] and not (set(e["avoid"]) & bad)
          and e["id"] not in used_today]
    if not ok:
        return None
    ok.sort(key=lambda e: used_count.get(e["id"], 0))     # sort เสถียร: ท่าหลักหลายข้อต่อมาก่อนเสมอ
    return ok[0]

def make_plan(p):
    days_n = min(max(int(p["days_per_week"]), 2), 5)
    split = SPLITS[days_n]
    if p["level"] == "beginner" and days_n == 3:
        split = ["FULL_A", "FULL_B", "FULL_A"]            # มือใหม่ 3 วัน เล่นทั้งตัวดีกว่า
    focus = list(p.get("target_muscles", []))
    used_count, days = {}, []
    for i, key in enumerate(split):
        title, slots = DAY_TYPES[key]
        slots = list(slots)
        for m in focus:                                    # ให้กล้ามเนื้อที่เน้นได้เล่นทุกวัน
            if m not in slots and len(slots) < 7:
                slots.append(m)
        used_today, items = set(), []
        for m in slots:
            ex = pick_exercise(m, p, used_today, used_count)
            if ex is None:
                continue
            used_today.add(ex["id"]); used_count[ex["id"]] = used_count.get(ex["id"], 0) + 1
            sets, reps, rest = prescription(ex, p, focus)
            items.append(dict(ex=ex, sets=sets, reps=reps, rest=rest, load=suggest_load(ex, p),
                              minutes=minutes_for(ex, sets, reps, rest)))
        total = math.ceil(sum(it["minutes"] for it in items)) + 5    # +5 นาทีวอร์มอัพ
        days.append(dict(weekday=WEEK[TRAIN_DAYS[days_n][i]], title=title, items=items, total_min=total))
    rest_days = [w for j, w in enumerate(WEEK) if j not in TRAIN_DAYS[days_n]]
    return dict(days=days, rest_days=rest_days, days_n=days_n)

def fmt_load(ex, load):
    if load is None:
        return "น้ำหนักตัว"
    x = int(load) if float(load).is_integer() else load
    if ex["equip"] == "dumbbell":
        return f"{x} กก. (อันเดียว)" if ex["id"] in SINGLE_DB else f"{x} กก./ข้าง"
    if ex["equip"] == "barbell":
        return f"{x} กก. (รวมบาร์)"
    return f"{x} กก. (ตุ้มน้ำหนัก)"

def fmt_reps(ex, reps):
    return f"{reps[0]}–{reps[1]} {'วินาที' if ex['unit'] == 'sec' else 'ครั้ง'}"

def render_plan(plan, p):
    """ตารางแผน (สร้างด้วยโค้ด ตัวเลขจึงไม่เพี้ยน)"""
    out = [f"## 📅 แผนออกกำลังกาย {plan['days_n']} วันต่อสัปดาห์\n"]
    for d in plan["days"]:
        out.append(f"### วัน{d['weekday']} · {d['title']} · ราว {d['total_min']} นาที (รวมวอร์มอัพ 5 นาที)")
        out.append("| ท่า | อุปกรณ์ | น้ำหนักเริ่มต้น | เซ็ต × ครั้ง | พัก | เวลา |")
        out.append("|---|---|---|---|---|---|")
        for it in d["items"]:
            ex = it["ex"]
            out.append(f"| {ex['th']} ({ex['en']}) | {EQUIP_TH[ex['equip']]} | {fmt_load(ex, it['load'])} | "
                       f"{it['sets']} × {fmt_reps(ex, it['reps'])} | {it['rest']} วิ | ~{math.ceil(it['minutes'])} นาที |")
        out.append("")
    out.append(f"**วันพัก:** {', '.join(plan['rest_days'])}\n")
    notes = ["วอร์มอัพ 5 นาที (เดินเร็วหรือปั่นจักรยานเบา ๆ + เคลื่อนไหวข้อต่อ) ก่อนเริ่มทุกครั้ง",
             "น้ำหนักในตารางเป็น**ค่าประมาณตอนเริ่มต้น** ให้เลือกน้ำหนักที่ทำครบจำนวนครั้งด้วยท่าที่ถูกต้อง และรู้สึกว่ายังเหลือแรงอีกราว 2 ครั้ง",
             "ถ้าทำได้ถึงจำนวนครั้งสูงสุดทุกเซ็ตติดกัน 2 สัปดาห์ ให้เพิ่มน้ำหนักทีละนิด (ประมาณ 2.5–5%)"]
    if p["injuries"]:
        notes.append(f"คุณมีอาการเจ็บบริเวณ {', '.join(p['injuries'])} เราเลี่ยงท่าที่กระทบแล้ว แต่นี่ไม่ใช่การรักษา ควรปรึกษาแพทย์หรือนักกายภาพก่อนเริ่มเล่น")
    if p["age"] >= 50 or p["age"] < 16:
        notes.append("ช่วงอายุของคุณควรให้แพทย์หรือเทรนเนอร์ดูท่าทางและน้ำหนักก่อนเริ่มจริง")
    if p.get("days_requested", 0) > 5:
        notes.append("คุณว่างเล่นเกิน 5 วัน แต่แนะนำเล่นไม่เกิน 5 วันและพักอย่างน้อย 2 วัน เพื่อให้กล้ามเนื้อฟื้นตัว")
    out.append("**ข้อควรรู้**\n" + "\n".join(f"- {n}" for n in notes))
    return "\n".join(out)

def plan_to_text(plan):
    """แผนแบบข้อความสั้น ใช้ใส่ซองให้โค้ชอ่านตอนตอบคำถามต่อ"""
    lines = []
    for d in plan["days"]:
        lines.append(f"วัน{d['weekday']} ({d['title']}) ~{d['total_min']} นาที")
        for it in d["items"]:
            ex = it["ex"]
            lines.append(f"  - {ex['th']} | {fmt_load(ex, it['load'])} | {it['sets']} เซ็ต × {fmt_reps(ex, it['reps'])} | พัก {it['rest']} วิ")
    lines.append("วันพัก: " + ", ".join(plan["rest_days"]))
    return "\n".join(lines)

### ทดสอบเครื่องมือก่อน (ยังไม่เรียก AI ไม่เสียโควตา)

ลองกรณีตัวอย่างของคุณ: ผู้ชายน้ำหนัก 70 กก. มือใหม่ อยากเพิ่มกล้ามแขน เล่นฟิตเนส 3 วันต่อสัปดาห์

In [5]:
demo = dict(goal="muscle", target_muscles=["biceps", "triceps"], weight_kg=70, age=25, sex="male",
            level="beginner", equipment="gym", days_per_week=3, injuries=[])
from IPython.display import Markdown, display
display(Markdown(render_plan(make_plan(demo), demo)))

## 📅 แผนออกกำลังกาย 3 วันต่อสัปดาห์

### วันจันทร์ · ทั้งตัว A · ราว 48 นาที (รวมวอร์มอัพ 5 นาที)
| ท่า | อุปกรณ์ | น้ำหนักเริ่มต้น | เซ็ต × ครั้ง | พัก | เวลา |
|---|---|---|---|---|---|
| เลกเพรส (Leg Press) | เครื่อง | 70 กก. (ตุ้มน้ำหนัก) | 3 × 8–12 ครั้ง | 75 วิ | ~6 นาที |
| เครื่องเชสต์เพรส (Machine Chest Press) | เครื่อง | 25 กก. (ตุ้มน้ำหนัก) | 3 × 8–12 ครั้ง | 75 วิ | ~6 นาที |
| แลทพูลดาวน์ (Lat Pulldown) | เครื่อง | 35 กก. (ตุ้มน้ำหนัก) | 3 × 8–12 ครั้ง | 75 วิ | ~6 นาที |
| เครื่องช็อลเดอร์เพรส (Machine Shoulder Press) | เครื่อง | 20 กก. (ตุ้มน้ำหนัก) | 3 × 8–12 ครั้ง | 75 วิ | ~6 นาที |
| แพลงก์ (Plank) | น้ำหนักตัว | น้ำหนักตัว | 3 × 20–45 วินาที | 45 วิ | ~4 นาที |
| ดัมเบลเคิร์ล (Dumbbell Curl) | ดัมเบล | 7 กก./ข้าง | 4 × 8–12 ครั้ง | 75 วิ | ~8 นาที |
| เคเบิลพุชดาวน์ (Triceps Pushdown) | สายเคเบิล | 17.5 กก. (ตุ้มน้ำหนัก) | 4 × 8–12 ครั้ง | 75 วิ | ~8 นาที |

### วันพุธ · ทั้งตัว B · ราว 38 นาที (รวมวอร์มอัพ 5 นาที)
| ท่า | อุปกรณ์ | น้ำหนักเริ่มต้น | เซ็ต × ครั้ง | พัก | เวลา |
|---|---|---|---|---|---|
| ไลอิ้งเลกเคิร์ล (Lying Leg Curl) | เครื่อง | 17.5 กก. (ตุ้มน้ำหนัก) | 3 × 8–12 ครั้ง | 75 วิ | ~6 นาที |
| ดัมเบลเบนช์เพรส (Dumbbell Bench Press) | ดัมเบล | 10 กก./ข้าง | 3 × 8–12 ครั้ง | 75 วิ | ~6 นาที |
| ซีทเต็ดเคเบิลโรว์ (Seated Cable Row) | สายเคเบิล | 32.5 กก. (ตุ้มน้ำหนัก) | 3 × 8–12 ครั้ง | 75 วิ | ~6 นาที |
| แฮมเมอร์เคิร์ล (Hammer Curl) | ดัมเบล | 7 กก./ข้าง | 4 × 8–12 ครั้ง | 75 วิ | ~8 นาที |
| ดัมเบลโอเวอร์เฮดไตรเซ็ปเอ็กซ์เทนชัน (Overhead Dumbbell Triceps Extension) | ดัมเบล | 10 กก. (อันเดียว) | 4 × 8–12 ครั้ง | 75 วิ | ~8 นาที |

### วันศุกร์ · ทั้งตัว A · ราว 50 นาที (รวมวอร์มอัพ 5 นาที)
| ท่า | อุปกรณ์ | น้ำหนักเริ่มต้น | เซ็ต × ครั้ง | พัก | เวลา |
|---|---|---|---|---|---|
| ก็อบเล็ตสควอท (Goblet Squat) | ดัมเบล | 14 กก. (อันเดียว) | 3 × 8–12 ครั้ง | 75 วิ | ~6 นาที |
| วิดพื้น (Push-up) | น้ำหนักตัว | น้ำหนักตัว | 3 × 8–12 ครั้ง | 75 วิ | ~6 นาที |
| ดัมเบลโรว์แขนเดียว (One-Arm Dumbbell Row) | ดัมเบล | 10 กก./ข้าง | 3 × 8–12 ครั้ง | 75 วิ | ~6 นาที |
| ดัมเบลช็อลเดอร์เพรส (Dumbbell Shoulder Press) | ดัมเบล | 8 กก./ข้าง | 3 × 8–12 ครั้ง | 75 วิ | ~6 นาที |
| เดดบัก (Dead Bug) | น้ำหนักตัว | น้ำหนักตัว | 3 × 8–12 ครั้ง | 75 วิ | ~6 นาที |
| เคเบิลเคิร์ล (Cable Curl) | สายเคเบิล | 15 กก. (ตุ้มน้ำหนัก) | 4 × 8–12 ครั้ง | 75 วิ | ~8 นาที |
| วิดพื้นมือชิด (Close-grip Push-up) | น้ำหนักตัว | น้ำหนักตัว | 4 × 8–12 ครั้ง | 75 วิ | ~8 นาที |

**วันพัก:** อังคาร, พฤหัสบดี, เสาร์, อาทิตย์

**ข้อควรรู้**
- วอร์มอัพ 5 นาที (เดินเร็วหรือปั่นจักรยานเบา ๆ + เคลื่อนไหวข้อต่อ) ก่อนเริ่มทุกครั้ง
- น้ำหนักในตารางเป็น**ค่าประมาณตอนเริ่มต้น** ให้เลือกน้ำหนักที่ทำครบจำนวนครั้งด้วยท่าที่ถูกต้อง และรู้สึกว่ายังเหลือแรงอีกราว 2 ครั้ง
- ถ้าทำได้ถึงจำนวนครั้งสูงสุดทุกเซ็ตติดกัน 2 สัปดาห์ ให้เพิ่มน้ำหนักทีละนิด (ประมาณ 2.5–5%)

**สิ่งที่จะเห็น:** ตาราง 3 วัน (ทั้งตัว A / B / A) ที่มีท่าแขนทุกวัน เช่น ดัมเบลเคิร์ล น้ำหนักเริ่มต้น **7 กก./ข้าง** เซ็ตละ 8–12 ครั้ง และมีเซ็ตมากกว่าท่าอื่น 1 เซ็ต

**ลองเล่นต่อ:** เปลี่ยน `sex="female"`, `weight_kg=55`, `equipment="bodyweight"` หรือ `injuries=["shoulder"]` แล้วดูว่าท่าและน้ำหนักเปลี่ยนอย่างไร

## ขั้น 3 สมุดบันทึกโปรไฟล์ (structured memory)

LLM ทำหน้าที่ **ดึงข้อมูลจากภาษาคน → JSON** (เช่น "ผมหนัก 70 อายุ 25" → `{"weight_kg": 70, "age": 25}`) แล้วโค้ดตรวจความสมเหตุสมผลของค่าก่อนเก็บ
โค้ดจะรู้เองว่ายังขาดช่องไหน และบังคับให้บอท**ถามก่อน** ไม่ให้ข้ามไปแนะนำเอง

In [6]:
MUSCLES = list(MUSCLE_TH)
REQUIRED = ["goal", "target_muscles", "weight_kg", "age", "sex", "level", "equipment", "days_per_week", "injuries"]
ASK_LABELS = {
    "goal": "เป้าหมาย (เพิ่มกล้ามเนื้อ / ลดไขมัน / เพิ่มความแข็งแรง)",
    "target_muscles": "อยากเน้นกล้ามเนื้อส่วนไหนเป็นพิเศษ (เช่น แขน อก ขา หรือทั้งตัว)",
    "weight_kg": "น้ำหนักตัวกี่กิโลกรัม (ไว้คำนวณน้ำหนักที่ยกให้พอดีกับคุณ)",
    "age": "อายุเท่าไร",
    "sex": "เพศ ชายหรือหญิง (ใช้ประมาณน้ำหนักเริ่มต้น)",
    "level": "ประสบการณ์ (มือใหม่ / เล่นมา 6 เดือน–2 ปี / เกิน 2 ปี)",
    "equipment": "เล่นที่ไหน มีอุปกรณ์อะไร (ฟิตเนสเครื่องครบ / ดัมเบลที่บ้าน / ไม่มีอุปกรณ์)",
    "days_per_week": "ว่างเล่นได้กี่วันต่อสัปดาห์",
    "injuries": "มีอาการบาดเจ็บหรือเจ็บส่วนไหนอยู่ไหม (ไหล่ เข่า หลัง ข้อมือ ข้อศอก หรือไม่มี)",
}
PROFILE_TH = {"goal": {"muscle": "เพิ่มกล้ามเนื้อ", "fat_loss": "ลดไขมัน", "strength": "เพิ่มความแข็งแรง"},
              "sex": {"male": "ชาย", "female": "หญิง"},
              "level": {"beginner": "มือใหม่", "intermediate": "ระดับกลาง", "advanced": "ระดับสูง"},
              "equipment": {"gym": "ฟิตเนสเครื่องครบ", "dumbbell": "ดัมเบลที่บ้าน", "bodyweight": "ไม่มีอุปกรณ์"}}

EXTRACT_RULES = """คุณคือตัวดึงข้อมูลโปรไฟล์ผู้ออกกำลังกายจากข้อความ ตอบเป็น JSON เท่านั้น ห้ามมีข้อความอื่น
ดึงเฉพาะสิ่งที่ผู้ใช้พูดถึงในข้อความล่าสุด ถ้าไม่ได้พูดถึงให้ไม่ต้องใส่ key นั้น
key ที่ใช้ได้:
- goal: "muscle" (เพิ่มกล้ามเนื้อ) | "fat_loss" (ลดไขมัน/ลดน้ำหนัก) | "strength" (เพิ่มความแข็งแรง)
- target_muscles: list ของ chest, back, shoulders, biceps, triceps, quads, hamstrings, calves, core
  (แขน = ["biceps","triceps"], กล้ามแขน/ไบเซ็ป = ["biceps"], ขา = ["quads","hamstrings"], ทั้งตัวหรือไม่เจาะจง = [])
- weight_kg: น้ำหนักตัวเป็นกิโลกรัม (ตัวเลข)
- age: อายุ (ตัวเลข)
- sex: "male" | "female"
- level: "beginner" (มือใหม่ เล่นน้อยกว่า 6 เดือน) | "intermediate" (6 เดือนถึง 2 ปี) | "advanced" (เกิน 2 ปี)
- equipment: "gym" (ฟิตเนสมีเครื่องครบ) | "dumbbell" (มีแค่ดัมเบลที่บ้าน) | "bodyweight" (ไม่มีอุปกรณ์)
- days_per_week: จำนวนวันที่ว่างเล่นต่อสัปดาห์ (ตัวเลข)
- injuries: list ของ shoulder, knee, back, wrist, elbow (ไม่มีอาการเจ็บ = [])
ผู้ใช้อาจตอบสั้น ๆ เช่น "70" หรือ "25 ชาย" ให้ดูจาก "คำถามล่าสุดของบอท" ประกอบ
ตัวอย่าง: "อยากเพิ่มกล้ามแขน" -> {"goal":"muscle","target_muscles":["biceps","triceps"]}"""

def extract_profile(message, last_bot, profile):
    user = f"คำถามล่าสุดของบอท: {last_bot or '(ยังไม่มี)'}\nข้อความล่าสุดของผู้ใช้: {message}"
    raw = call_llm([{"role": "system", "content": EXTRACT_RULES}, {"role": "user", "content": user}], temperature=0)
    m = re.search(r"\{.*\}", raw, re.S)
    try:
        return json.loads(m.group()) if m else {}
    except json.JSONDecodeError:
        return {}

def clean_updates(raw):
    """ตรวจค่าที่ LLM ดึงมา เก็บเฉพาะที่สมเหตุสมผล (กัน LLM เดาตัวเลขเพี้ยน)"""
    u = {}
    def num(v):
        try: return float(v)
        except (TypeError, ValueError): return None
    if raw.get("goal") in PROFILE_TH["goal"]: u["goal"] = raw["goal"]
    if isinstance(raw.get("target_muscles"), list):
        u["target_muscles"] = [m for m in raw["target_muscles"] if m in MUSCLES]
    w = num(raw.get("weight_kg"))
    if w and 30 <= w <= 250: u["weight_kg"] = w
    a = num(raw.get("age"))
    if a and 10 <= a <= 90: u["age"] = int(a)
    if raw.get("sex") in PROFILE_TH["sex"]: u["sex"] = raw["sex"]
    if raw.get("level") in PROFILE_TH["level"]: u["level"] = raw["level"]
    if raw.get("equipment") in PROFILE_TH["equipment"]: u["equipment"] = raw["equipment"]
    d = num(raw.get("days_per_week"))
    if d and 1 <= d <= 7:
        u["days_requested"] = int(d)
        u["days_per_week"] = min(max(int(d), 2), 5)
    if isinstance(raw.get("injuries"), list):
        u["injuries"] = [i for i in raw["injuries"] if i in ("shoulder", "knee", "back", "wrist", "elbow")]
    return u

def profile_text(p):
    if not p: return "(ยังไม่มีข้อมูล)"
    parts = []
    for k, v in p.items():
        if k == "days_requested": continue
        if k in PROFILE_TH: v = PROFILE_TH[k].get(v, v)
        elif k == "target_muscles": v = ", ".join(MUSCLE_TH[m] for m in v) or "ทั้งตัว"
        elif k == "injuries": v = ", ".join(v) or "ไม่มี"
        parts.append(f"{k}: {v}")
    return " | ".join(parts)

print(clean_updates({"weight_kg": "70", "age": 25, "sex": "male", "goal": "muscle", "level": "pro", "days_per_week": 7}))

{'goal': 'muscle', 'weight_kg': 70.0, 'age': 25, 'sex': 'male', 'days_requested': 7, 'days_per_week': 5}


**สิ่งที่จะเห็น:** `level: "pro"` ถูกตัดทิ้งเพราะไม่ใช่ค่าที่ระบบรู้จัก และ `days_per_week: 7` ถูกบีบเหลือ 5 (เก็บค่าที่ขอไว้ใน `days_requested` เพื่อเตือนเรื่องการพัก)

## ขั้น 4 กฎของโค้ช และการประกอบร่างแชทบอท

ในแต่ละเทิร์นบอทจะ **① ดึงข้อมูลใหม่ใส่โปรไฟล์ → ② ถ้ายังขาดช่อง ถามต่ออีกไม่เกิน 3 ข้อ → ③ ถ้าครบ สร้างแผนด้วยเครื่องมือแล้วให้ LLM เขียนคำนำ → ④ เทิร์นถัด ๆ ไปตอบคำถามต่อโดยอ้างอิงแผนและการ์ดท่า**
ถ้าผู้ใช้เปลี่ยนข้อมูล (เช่น "เปลี่ยนเป็นเล่น 4 วัน" หรือ "ตอนนี้เจ็บไหล่") แผนจะถูกคำนวณใหม่ให้ทันที

In [7]:
COACH_RULES = """คุณคือ "Stronger Assistant" โค้ชออกกำลังกายส่วนตัว
- ตอบภาษาไทย เป็นกันเอง กระชับ ให้กำลังใจ ไม่ต้องทักทายซ้ำทุกครั้ง
- ห้ามแนะนำท่า น้ำหนัก จำนวนเซ็ตหรือครั้ง จนกว่าจะรู้ข้อมูลผู้ใช้ครบ ต้องถามก่อนเสมอ
- ตัวเลขน้ำหนัก เซ็ต ครั้ง เวลาพัก ต้องใช้จาก [ข้อมูลอ้างอิง] เท่านั้น ห้ามคิดเลขเองหรือแก้ตัวเลข
- ถ้าผู้ใช้ถามเรื่องที่ไม่อยู่ในข้อมูลอ้างอิง และเป็นความรู้ออกกำลังกายทั่วไป ตอบได้ แต่ห้ามให้ตัวเลขน้ำหนักเฉพาะเจาะจง
- ถ้ามีอาการเจ็บหน้าอก เวียนหัว หายใจไม่ออก หรือเจ็บรุนแรง ให้บอกให้หยุดเล่นและพบแพทย์
- ห้ามวินิจฉัยโรค ห้ามแนะนำยาหรืออาหารเสริม ห้ามสนับสนุนการอดอาหารหรือลดน้ำหนักแบบสุดโต่ง"""

def ask_missing(message, history, profile, missing):
    ask = missing[:3]                                         # ถามทีละไม่เกิน 3 ข้อ ไม่ให้ผู้ใช้เหนื่อย
    task = ("\n\n[งานตอนนี้] ข้อมูลของผู้ใช้ยังไม่ครบ ห้ามแนะนำท่า น้ำหนัก เซ็ต หรือจำนวนครั้ง "
            "ให้ทักทายสั้น ๆ (ถ้ายังไม่เคยคุย) บอกสั้น ๆ ว่าต้องถามเพื่อให้แผนเหมาะกับตัวเขา แล้วถามเฉพาะข้อต่อไปนี้ เป็นข้อ ๆ:\n"
            + "\n".join(f"- {ASK_LABELS[k]}" for k in ask))
    env = build_envelope(message, COACH_RULES + task, memory=profile_text(profile), history=history)
    return call_llm(env, temperature=0.3)

def plan_intro(profile, plan):
    task = ("\n\n[งานตอนนี้] เขียนคำนำสั้น ๆ 2–3 ประโยคก่อนแสดงแผน บอกว่าออกแบบจากข้อมูลไหนของผู้ใช้ "
            "(เป้าหมาย ระดับ อุปกรณ์ จำนวนวัน ส่วนที่เน้น) ห้ามใส่ตัวเลขน้ำหนัก เซ็ต หรือจำนวนครั้ง "
            "และห้ามสร้างตารางเอง เพราะระบบจะแนบตารางแผนต่อท้ายให้")
    env = build_envelope("ช่วยสรุปคำนำของแผนให้หน่อย", COACH_RULES + task, memory=profile_text(profile))
    return call_llm(env, temperature=0.3)

def cue_cards(message, plan, limit=6):
    """หยิบการ์ดท่า: ถ้าผู้ใช้เอ่ยชื่อท่า ให้หยิบท่านั้น ไม่งั้นหยิบท่าทั้งหมดในแผน"""
    low = message.lower()
    hits = [e for e in EXERCISES if e["th"] in message or e["en"].lower() in low]
    if not hits:
        seen = {}
        for d in plan["days"]:
            for it in d["items"]:
                seen.setdefault(it["ex"]["id"], it["ex"])
        hits = list(seen.values())
    return "\n\n".join(f"[{e['th']} ({e['en']}) | เน้น: {MUSCLE_TH[e['muscle']]} | อุปกรณ์: {EQUIP_TH[e['equip']]}]\n"
                        f"วิธีทำ: {e['cue']}" for e in hits[:limit])

def answer_question(message, history, profile, plan):
    cards = "[แผนของผู้ใช้]\n" + plan_to_text(plan) + "\n\n[การ์ดท่า]\n" + cue_cards(message, plan)
    env = build_envelope(message, COACH_RULES, memory=profile_text(profile), cards=cards, history=history)
    return call_llm(env, temperature=0.3)

SESSIONS = {}   # เก็บโปรไฟล์และแผนแยกตามผู้ใช้ (session)

def coach_reply(message, history, sid="local"):
    S = SESSIONS.setdefault(sid, {"profile": {}, "plan": None})
    profile = S["profile"]
    last_bot = next((h["content"] for h in reversed(history) if h["role"] == "assistant"), "")
    updates = clean_updates(extract_profile(message, last_bot, profile))
    changed = {k: v for k, v in updates.items() if profile.get(k) != v and k != "days_requested"}
    profile.update(updates)

    missing = [k for k in REQUIRED if k not in profile]
    if missing:                                               # ① ยังไม่ครบ → ถามก่อน
        return ask_missing(message, history, profile, missing)
    if S["plan"] is None or changed:                          # ② ครบแล้ว (หรือผู้ใช้แก้ข้อมูล) → สร้างแผนใหม่
        S["plan"] = make_plan(profile)
        return plan_intro(profile, S["plan"]) + "\n\n" + render_plan(S["plan"], profile)
    return answer_question(message, history, profile, S["plan"])   # ③ ตอบคำถามต่อ

def reset_session(sid="local"):
    SESSIONS.pop(sid, None)

## ขั้น 5 ลองคุยในคอนโซล

จำลองบทสนทนาตามตัวอย่างของคุณ ตรวจดูว่าบอท **ถามก่อน** แล้วค่อยแนะนำ และ **ตอบคำถามต่อ** ได้

In [8]:
reset_session()
history = []
for user_msg in ["อยากเพิ่มกล้ามเนื้อแขน",
                 "น้ำหนัก 70 กิโล อายุ 25 ผู้ชาย",
                 "มือใหม่ เล่นที่ฟิตเนส ว่าง 3 วัน ไม่เจ็บตรงไหนเลย",
                 "ดัมเบลเคิร์ลทำยังไงให้ถูกท่า"]:
    reply = coach_reply(user_msg, history)
    print(f"🧑 {user_msg}\n🤖 {reply}\n{'-'*60}")
    history += [{"role": "user", "content": user_msg}, {"role": "assistant", "content": reply}]

🧑 อยากเพิ่มกล้ามเนื้อแขน
🤖 ยอดเยี่ยมเลยครับ! เป้าหมายเพิ่มกล้ามเนื้อต้นแขนหน้าและแขนหลังนี่ลุยกันได้เต็มที่เลยครับ 

เพื่อให้ผมจัดแผนการออกกำลังกายให้เหมาะกับตัวคุณที่สุดและปลอดภัยที่สุด รบกวนขอข้อมูล 3 ข้อนี้นะครับ:

1. น้ำหนักตัวกี่กิโลกรัมครับ?
2. อายุเท่าไรครับ?
3. เพศ ชายหรือหญิงครับ?
------------------------------------------------------------
🧑 น้ำหนัก 70 กิโล อายุ 25 ผู้ชาย
🤖 ข้อมูลครบถ้วนแล้วครับ เยี่ยมเลย! 

เพื่อให้ผมจัดตารางและเลือกท่าทางที่เหมาะสมกับความพร้อมของคุณที่สุด ขอข้อมูลเพิ่มอีกนิดนะครับ:

1. ประสบการณ์การออกกำลังกาย (มือใหม่ / เล่นมา 6 เดือน–2 ปี / เกิน 2 ปี)
2. สถานที่และอุปกรณ์ (ฟิตเนสเครื่องครบ / ดัมเบลที่บ้าน / ไม่มีอุปกรณ์)
3. จำนวนวันที่สะดวกเล่นต่อสัปดาห์ครับ
------------------------------------------------------------
🧑 มือใหม่ เล่นที่ฟิตเนส ว่าง 3 วัน ไม่เจ็บตรงไหนเลย
🤖 นี่คือแผนสำหรับคุณชาย อายุ 25 ปี ระดับมือใหม่ ที่ตั้งใจมาลุยสร้างกล้ามเนื้อต้นแขนทั้งด้านหน้าและด้านหลังให้แน่น ๆ ครับ โดยออกแบบให้เหมาะกับอุปกรณ์ฟิตเนสแบบครบครัน 3 วันต่อสัปดาห์ ลุยกันเล

**สิ่งที่จะเห็น**
- เทิร์น 1–2: บอทถามข้อมูลที่ยังขาด **ไม่มีตัวเลขน้ำหนักหรือเซ็ต**
- เทิร์น 3: ได้แผน 3 วันครบ โดยดัมเบลเคิร์ลเริ่มที่ 7 กก./ข้าง
- เทิร์น 4: ตอบวิธีทำท่าเคิร์ลจากการ์ด และยังใช้ตัวเลขจากแผนเดิม

**ลองเล่นต่อ:** พิมพ์ "เปลี่ยนเป็นเล่น 4 วัน" หรือ "ตอนนี้เจ็บข้อศอก" แล้วดูว่าแผนถูกคำนวณใหม่

## ขั้น 6 ข้อสอบกันพลาด (แนวข้อสอบชุดทองของบิ๊บ)

ตรวจกติกาสำคัญที่สุดของบอทตัวนี้ คือ **ห้ามแนะนำก่อนถาม** และ **ตัวเลขต้องตรงกับเครื่องมือ**

In [9]:
def run_case(msgs, sid="test"):
    reset_session(sid); hist, replies = [], []
    for m in msgs:
        r = coach_reply(m, hist, sid=sid)
        replies.append(r); hist += [{"role": "user", "content": m}, {"role": "assistant", "content": r}]
    return replies

NUM_PAT = re.compile(r"\d+\s*(กก|กิโล|kg|เซ็ต|ครั้ง)")
results = []

# ข้อ 1: ข้อมูลน้อย ต้องถามกลับ ไม่มีตัวเลขแนะนำ
r = run_case(["อยากเพิ่มกล้ามแขน ควรยกดัมเบลกี่โล"])[0]
results.append(("ถามก่อนแนะนำ (ไม่มีตัวเลขน้ำหนัก/เซ็ต และยังไม่สร้างแผน)", SESSIONS["test"]["plan"] is None, not NUM_PAT.search(r)))

# ข้อ 2: ค่าที่เป็นไปไม่ได้ต้องไม่ถูกเก็บ
results.append(("ตัดน้ำหนัก 5 กก. ทิ้ง", clean_updates({"weight_kg": 5}) == {}, True))

# ข้อ 3: น้ำหนักในแผนตรงกับเครื่องมือ (7 กก./ข้างสำหรับ 70 กก. มือใหม่)
ex = next(e for e in EXERCISES if e["id"] == "db_curl")
results.append(("ดัมเบลเคิร์ล 70 กก. มือใหม่ = 7 กก.", suggest_load(ex, demo) == 7, True))

# ข้อ 4: บาดเจ็บไหล่ ต้องไม่มีท่าที่ระบุว่ากระทบไหล่
plan = make_plan({**demo, "injuries": ["shoulder"]})
bad = [it["ex"]["th"] for d in plan["days"] for it in d["items"] if "shoulder" in it["ex"]["avoid"]]
results.append(("เจ็บไหล่ → ไม่มีท่าที่กระทบไหล่", not bad, True))

# ข้อ 5: ไม่มีอุปกรณ์ ต้องมีแต่ท่าน้ำหนักตัว
plan = make_plan({**demo, "equipment": "bodyweight"})
eq = {it["ex"]["equip"] for d in plan["days"] for it in d["items"]}
results.append(("ไม่มีอุปกรณ์ → ท่าน้ำหนักตัวล้วน", eq == {"bodyweight"}, True))

import pandas as pd
df = pd.DataFrame([{"ข้อสอบ": n, "ผ่าน": bool(a and b)} for n, a, b in results])
print(f"ผ่าน {df['ผ่าน'].sum()}/{len(df)} ข้อ")
df

ผ่าน 5/5 ข้อ


,ข้อสอบ,ผ่าน
0,ถามก่อนแนะนำ (ไม่มีตัวเลขน้ำหนัก/เซ็ต และยังไม...,True
1,ตัดน้ำหนัก 5 กก. ทิ้ง,True
2,ดัมเบลเคิร์ล 70 กก. มือใหม่ = 7 กก.,True
3,เจ็บไหล่ → ไม่มีท่าที่กระทบไหล่,True
4,ไม่มีอุปกรณ์ → ท่าน้ำหนักตัวล้วน,True


## ขั้น 7 เปิดเป็นหน้าแชทให้เพื่อนลอง (Gradio)

รันแล้วจะได้ลิงก์ `gradio.live` ชั่วคราว ส่งให้เพื่อนลองได้ แต่ละคนมีโปรไฟล์ของตัวเองแยกกัน (แยกตาม session)

In [10]:
import gradio as gr

def gradio_chat(message, history, request: gr.Request):
    hist = []
    for h in history:
        if isinstance(h, dict):
            hist.append({"role": h["role"], "content": str(h["content"])})
        else:
            hist += [{"role": "user", "content": str(h[0])}, {"role": "assistant", "content": str(h[1])}]
    return coach_reply(message, hist, sid=request.session_hash)

gr.ChatInterface(
    gradio_chat,
    title="💪 Stronger Assistant",
    description="โค้ชออกกำลังกายส่วนตัว บอกเป้าหมายได้เลย เช่น \"อยากเพิ่มกล้ามแขน\" แล้วโค้ชจะถามข้อมูลก่อนแนะนำ",
    examples=["อยากเพิ่มกล้ามแขน", "อยากลดพุง มีดัมเบลที่บ้าน", "อยากแข็งแรงขึ้น เล่นที่ฟิตเนส"],
).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b974e50e2d91575fbc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## ต่อยอดต่อได้

- เพิ่มท่าใน `EXERCISES` ได้เรื่อย ๆ (เพิ่มแค่ 1 บรรทัด `E(...)`) ระบบจะหยิบไปใช้เอง
- เปลี่ยนฐานความรู้เป็น embedding + Chroma ตามบทที่ 4 ของอาจารย์ได้ ถ้ามีบทความหรือคู่มือท่าออกกำลังยาว ๆ
- เก็บโปรไฟล์ลงไฟล์ JSON หรือฐานข้อมูล เพื่อให้จำผู้ใช้ข้ามวัน
- เพิ่มบันทึกผลการเล่นจริง เพื่อให้บอทปรับน้ำหนักขึ้นตามความก้าวหน้า (progressive overload)
- ⚠️ บอทนี้ให้คำแนะนำทั่วไป ไม่ใช่คำแนะนำทางการแพทย์ ถ้ามีโรคประจำตัวหรืออาการเจ็บ ควรปรึกษาแพทย์หรือเทรนเนอร์